# Basic Convolution Neural Net Practice

Practicing conv nets from scratch on the MNIST greyscale number images for multiclass classification for a bit of practice

Following: https://victorzhou.com/blog/intro-to-cnns-part-1/#34-implementing-convolution 

In [8]:

import numpy as np
import idx2numpy
import pandas as pd
import random


## Setup

In [ ]:
# Using the Sobel filter for convolutions
sobel = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])
sobel

array([[ 1,  0, -1],
       [ 2,  0, -2],
       [ 1,  0, -1]])

## Convolution Net setup - Numpy

In [ ]:
class ConvNet:
    # Convolution Net with 3x3 filters
    def __init__(self, num_filters):
        self.num_filters = num_filters
        self.filters = np.random.rand(num_filters, 3, 3) / 9 # where filters is a 3d array with dimensions (num_filters, 3, 3) and divide by 9 to reduce variance
        
    def iterate_regions(self, image):
        # Generates all possible 3x3 image regions using valid padding
        h, w = image.shape
        
        for i in range(h - 2):
            for j in range(w - 2):
                im_region = image[i:(i + 3), j:(j + 3)]
                yield im_region, i, j
                
    def forward(self, input):
        # Performs a forward pass of the conv layer using the given input.
        # Returns a 3d numpy array with dimensions (h, w, num_filters).
        h, w = input.shape
        output = np.zeros((h - 2, w - 2, self.num_filters))
        
        for im_region, i, j in self.iterate_regions(input):
            output[i, j] = np.sum(im_region * self.filters, axis=(1, 2)) # This is where the convolutions happen
            
        return output
    

In [ ]:
class MaxPool2:
    # A Max Pooling layer using a pool size of 2.
    def iterate_regions(self, image):
        # Generates non-overlapping 2x2 image regions to pool over
        h, w, _ = image.shape
        
        new_h = h // 2
        new_w = w // 2
        
        for i in range(new_h):
            for j in range(new_w):
                im_region = image[(i * 2):(i * 2 + 2), (j * 2):(j * 2 + 2)]
                yield im_region, i, j
                
    def forward(self, input):
        # Performs a forward pass of the maxpool layer using the given input.
        # Returns a 3d numpy array with dimensions (h / 2, w / 2, num_filters).
        h, w, num_filters = input.shape
        output = np.zeros((h // 2, w // 2, num_filters))
        
        for im_region, i, j in self.iterate_regions(input):
            output[i, j] = np.amax(im_region, axis=(0, 1)) # This is where the max pooling happens
            
        return output

In [ ]:
class Softmax:
    # A standard fully-connected layer with softmax activation.
    def __init__(self, input_len, nodes):
        self.input_len = input_len
        self.nodes = nodes
        self.weights = np.random.rand(input_len, nodes) / input_len
        self.biases = np.random.rand(nodes) / input_len
        
    def forward(self, input):
        # Performs a forward pass of the softmax layer using the given input.
        # Returns a 1d numpy array containing probability values.
        input = input.flatten()
        totals = np.dot(input, self.weights) + self.biases
        exp = np.exp(totals)
        return exp / np.sum(exp, axis=0)

## Full network

In [ ]:
conv = ConvNet(8) # 8 filters
pool = MaxPool2()
softmax = Softmax(13 * 13 * 8, 10) # 13x13 is the output size of the conv layer, and 8 is the number of filters, and 10 is the number of classes for the output layer

def forward(image, label):
    out = conv.forward((image / 255) - 0.5)
    out = pool.forward(out)
    out = softmax.forward(out)
    
    loss = -np.log(out[label])
    acc = 1 if np.argmax(out) == label else 0
    
    return out, loss, acc

loss = 0
num_correct = 0
for i, (im, label) in enumerate(zip(train_images, train_labels)):
    _, l, acc = forward(im, label)
    loss += l
    num_correct += acc
    
    if i % 100 == 99:
        print('Step %d: Past 100 steps: Average Loss %.3f | Accuracy: %d%%' %
              (i + 1, loss / 100, num_correct))
        loss = 0
        num_correct = 0
        
        

## Now trying it on my own with PyTorch

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.utils.data import DataLoader
from torch.optim import Adam

In [ ]:
from torchvision import datasets, transforms


transform = transforms.Compose([transforms.ToTensor()])

train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='data', train=False, download=True, transform=transform)

train_loader = Data

100.0%
100.0%
100.0%
100.0%


### Set up the Conv Net